# Chest X-Ray Model Training Notebook
### MediScan AI Project

This notebook is for experimenting with the model before finalizing train.py

**Dataset:** Chest X-Ray Images (Pneumonia) - Kaggle

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

In [ ]:
# data transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# change path to your kaggle dataset
DATA_PATH = '../data/chest_xray/train'
dataset = datasets.ImageFolder(DATA_PATH, transform=transform)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

print('Classes:', dataset.classes)
print('Total images:', len(dataset))

In [ ]:
# load resnet18
model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# training loop - quick test with 2 epochs
for epoch in range(2):
    model.train()
    total_loss = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}')

In [ ]:
# visualize a batch
imgs, labels = next(iter(loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = imgs[i].permute(1, 2, 0).numpy()
    img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # denormalize
    ax.imshow(img.clip(0, 1))
    ax.set_title(dataset.classes[labels[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

### Note
For full training use `python ml/train.py` - its faster and saves the model automatically